In [2]:
# Cell 1: Setup - Đảm bảo working directory là segment4
import os
import sys
import logging

# Chuyển working directory về segment4 (quan trọng!)
os.chdir('/home/hieu0606sunny/price2026wsl/tech2ai/segment4')
sys.path.insert(0, '/home/hieu0606sunny/price2026wsl/tech2ai/segment4')

print(f"Working directory: {os.getcwd()}")

# Setup logging
logging.basicConfig(level=logging.INFO)
root = logging.getLogger()
root.setLevel(logging.INFO)

from dotenv import load_dotenv
load_dotenv(override=True)

# Check BRAVE_API_KEY
brave_key = os.getenv("BRAVE_API_KEY")
print(f"BRAVE_API_KEY: {'✅ Found' if brave_key else '❌ Not found'}")

Working directory: /home/hieu0606sunny/price2026wsl/tech2ai/segment4
BRAVE_API_KEY: ✅ Found


In [3]:
# Cell 2: Import dependencies
import asyncio
from typing import List

from openai import OpenAI
from pydantic import BaseModel, Field
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

print("✅ Imports successful!")

✅ Imports successful!


In [3]:
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

params =  {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}


In [16]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    brave = await server.list_tools()

brave

[Tool(name='brave_web_search', title=None, description='Performs a web search using the Brave Search API, ideal for general queries, news, articles, and online content. Use this for broad information gathering, recent events, or when you need diverse web sources. Supports pagination, content filtering, and freshness controls. Maximum 20 results per request, with offset for pagination. ', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query (max 400 chars, 50 words)'}, 'count': {'type': 'number', 'description': 'Number of results (1-20, default 10)', 'default': 10}, 'offset': {'type': 'number', 'description': 'Pagination offset (max 9, default 0)', 'default': 0}}, 'required': ['query']}, outputSchema=None, icons=None, annotations=None, meta=None),
 Tool(name='brave_local_search', title=None, description="Searches for local businesses and places using Brave's Local Search API. Best for queries related to physical locations, businesses, re

In [4]:
# Cell 3: Define SearchResults schema cho Amazon
# Giống như BestBuy nhưng cho Amazon

class SearchResults(BaseModel):
    """URLs found from Brave Search"""
    product_urls: List[str] = Field(description="List of Amazon product URLs")

print("✅ SearchResults schema defined!")
print(f"   Schema: {SearchResults.schema()}")

✅ SearchResults schema defined!
   Schema: {'description': 'URLs found from Brave Search', 'properties': {'product_urls': {'description': 'List of Amazon product URLs', 'items': {'type': 'string'}, 'title': 'Product Urls', 'type': 'array'}}, 'required': ['product_urls'], 'title': 'SearchResults', 'type': 'object'}


/tmp/ipykernel_20878/1984836345.py:9: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  print(f"   Schema: {SearchResults.schema()}")


In [5]:
# Cell 4: Define AmazonSearchAgent
# Tương tự BestBuySearchAgent nhưng điều chỉnh prompt cho Amazon

class AmazonSearchAgent:
    """
    Agent that uses Brave MCP Server to search for Amazon product URLs.
    
    This agent:
    1. Takes a keyword (e.g., "laptop")
    2. Uses Brave Search with site:amazon.com filter
    3. Returns a list of Amazon product URLs (format: /dp/ASIN or /gp/product/ASIN)
    """
    
    name = "Amazon Search Agent"
    MODEL = "gpt-5-nano"
    
    # INSTRUCTIONS cho việc tìm kiếm Amazon
    INSTRUCTIONS = """You are a web search agent. Your job is to find product URLs on Amazon.

STEPS:
1. Use brave_web_search with the given search query
2. Extract product URLs from results
3. Return ONLY URLs that are Amazon product pages

VALID Amazon product URL patterns (based on actual results):
- https://www.amazon.com/PRODUCT-NAME/dp/XXXXXXXXXX
- https://www.amazon.com/*/dp/XXXXXXXXXX
- Examples:
  - https://www.amazon.com/amazon-fire-tv-43-inch-4-series-4k-smart-tv/dp/B0CZ9WV2ZX
  - https://www.amazon.com/Apple-2025-MacBook-13-inch-Laptop/dp/B0DZD9S5GC
  - https://www.amazon.com/ASUS-Gaming-Laptop-Nebula-Display/dp/B0DW1X5YCQ

IMPORTANT:
- The ASIN code (after /dp/) is 10 characters (letters and numbers)
- Do NOT return search pages (amazon.com/s?k=...)
- Do NOT return category or brand pages
- Do NOT return Amazon homepage
- Return as many product URLs as you can find
"""
    
    def __init__(self):
        """Initialize the search agent."""
        logging.info(f"[{self.name}] Initializing...")
        self.brave_api_key = os.getenv("BRAVE_API_KEY")
        if not self.brave_api_key:
            raise ValueError("BRAVE_API_KEY not found in environment variables")
        logging.info(f"[{self.name}] Ready!")
    
    def get_brave_params(self) -> dict:
        """Get Brave MCP server parameters."""
        return {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-brave-search"],
            "env": {"BRAVE_API_KEY": self.brave_api_key}
        }
    
    async def _search_async(self, keyword: str, max_urls: int = 15) -> List[str]:
        """
        Async implementation of search.
        
        Args:
            keyword: Product keyword to search (e.g., "laptop")
            max_urls: Maximum number of URLs to return
            
        Returns:
            List of Amazon product URLs
        """
        logging.info(f"[{self.name}] Searching for: {keyword}")
        
        async with MCPServerStdio(
            params=self.get_brave_params(),
            client_session_timeout_seconds=60
        ) as brave_server:
            
            # Sử dụng trace để quan sát
            with trace(workflow_name="Amazon Search", group_id=keyword):
                search_agent = Agent(
                    name="AmazonSearchAgent",
                    instructions=self.INSTRUCTIONS,
                    model=self.MODEL,
                    mcp_servers=[brave_server],
                    output_type=SearchResults
                )
                
                result = await Runner.run(
                    search_agent,
                    f"Search for {keyword} on Amazon. Find product pages with /dp/ in the URL.",
                    max_turns=30
                )
                
                urls = result.final_output.product_urls[:max_urls]
                logging.info(f"[{self.name}] Found {len(urls)} product URLs")
                return urls
    
    def search(self, keyword: str, max_urls: int = 15) -> List[str]:
        """
        Search for Amazon products using Brave Search.
        
        Args:
            keyword: Product keyword to search (e.g., "laptop", "headphones")
            max_urls: Maximum number of URLs to return (default: 15)
            
        Returns:
            List of Amazon product URLs
        """
        try:
            loop = asyncio.get_running_loop()
        except RuntimeError:
            loop = asyncio.new_event_loop()
            asyncio.set_event_loop(loop)
            return loop.run_until_complete(self._search_async(keyword, max_urls))
        else:
            import nest_asyncio
            nest_asyncio.apply()
            return loop.run_until_complete(self._search_async(keyword, max_urls))


print("✅ AmazonSearchAgent class defined!")

✅ AmazonSearchAgent class defined!


In [ ]:
# Cell 5: Test AmazonSearchAgent với keyword "laptop"
# 🔍 Trace sẽ được ghi lại tại: https://platform.openai.com/traces

# Khởi tạo agent
search_agent = AmazonSearchAgent()

# Tìm kiếm sản phẩm
keyword = "Smart TV"
print(f"🔍 Searching for '{keyword}' on Amazon...")
print(f"   Using Brave MCP + GPT-5-nano")
print(f"   Trace: Check https://platform.openai.com/traces for details\n")

urls = search_agent.search(keyword, max_urls=10)

print(f"\n{'='*70}")
print(f"✅ Found {len(urls)} product URLs:")
print(f"{'='*70}\n")

for i, url in enumerate(urls, 1):
    # Highlight nếu URL có /dp/ (đúng format)
    is_valid = "/dp/" in url or "/gp/product/" in url
    status = "✓" if is_valid else "⚠️"
    print(f"{i}. {status} {url}")

INFO:root:[Amazon Search Agent] Initializing...
INFO:root:[Amazon Search Agent] Ready!
INFO:root:[Amazon Search Agent] Searching for: Smart TV


🔍 Searching for 'Smart TV' on Amazon...
   Using Brave MCP + GPT-5-nano
   Trace: Check https://platform.openai.com/traces for details



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:root:[Amazon Search Agent] Found 10 product URLs



✅ Found 10 product URLs:

1. ✓ https://www.amazon.com/amazon-fire-tv-43-inch-4-series-4k-smart-tv/dp/B0CZ9WV2ZX
2. ✓ https://www.amazon.com/amazon-fire-tv-32-inch-2-series-hd-smart-tv/dp/B0CJDSNN4T
3. ✓ https://www.amazon.com/introducing-amazon-fire-tv-32-inch-2-series-hd-smart-tv/dp/B09N6F9NV3
4. ✓ https://www.amazon.com/amazon-fire-tv-50-inch-4-series-4k-smart-tv/dp/B0CZBLZYY5
5. ✓ https://www.amazon.com/amazon-fire-tv-50-inch-omni-series-4k-smart-tv/dp/B08T6F8YBH
6. ✓ https://www.amazon.com/amazon-fire-tv-40-inch-2-series-hd-smart-tv/dp/B0CJCYMBZJ
7. ✓ https://www.amazon.com/amazon-fire-tv-55-inch-4-series-4k-smart-tv/dp/B08P3QB66R
8. ✓ https://www.amazon.com/Roku-Smart-2025-Television-Entertainment/dp/B0DWHKF7V5
9. ✓ https://www.amazon.com/Smart-TV-Amazon-Fire/dp/B071VRFFJ6
10. ✓ https://www.amazon.com/amazon-fire-tv-65-inch-omni-qled-series-smart-tv/dp/B0BJMGB9RN


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


In [7]:
# Cell 5: Test AmazonSearchAgent với keyword "laptop"
# 🔍 Trace sẽ được ghi lại tại: https://platform.openai.com/traces

# Khởi tạo agent
search_agent = AmazonSearchAgent()

# Tìm kiếm sản phẩm
keyword = "Laptop"
print(f"🔍 Searching for '{keyword}' on Amazon...")
print(f"   Using Brave MCP + GPT-5-nano")
print(f"   Trace: Check https://platform.openai.com/traces for details\n")

urls = search_agent.search(keyword, max_urls=10)

print(f"\n{'='*70}")
print(f"✅ Found {len(urls)} product URLs:")
print(f"{'='*70}\n")

for i, url in enumerate(urls, 1):
    # Highlight nếu URL có /dp/ (đúng format)
    is_valid = "/dp/" in url or "/gp/product/" in url
    status = "✓" if is_valid else "⚠️"
    print(f"{i}. {status} {url}")

INFO:root:[Amazon Search Agent] Initializing...
INFO:root:[Amazon Search Agent] Ready!
INFO:root:[Amazon Search Agent] Searching for: Laptop


🔍 Searching for 'Laptop' on Amazon...
   Using Brave MCP + GPT-5-nano
   Trace: Check https://platform.openai.com/traces for details



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:root:[Amazon Search Agent] Found 9 product URLs



✅ Found 9 product URLs:

1. ✓ https://www.amazon.com/DisplayPort-Adapter-Converter-Gold-Plated-Compatible/dp/B017Q8ZVWK
2. ✓ https://www.amazon.com/Cable-Matters-DisplayPort-Adapter-Resolution/dp/B00JQORLCG
3. ✓ https://www.amazon.com/DisplayPort-DP-HDMI-Adapter-Thunderbolt/dp/B0DSHY2LXP
4. ✓ https://www.amazon.com/Cable-Matters-DisplayPort-Feet-Resolution/dp/B005H3Q59U
5. ✓ https://www.amazon.com/DisplayPort-Capshi-Braided-Laptop-Monitor/dp/B07F7YV4BR
6. ✓ https://www.amazon.com/Docking-Station-Display-Windows-Ethernet/dp/B0CHHX7WYN
7. ✓ https://www.amazon.com/HP-Micro-edge-Microsoft-14-dq0040nr-Snowflake/dp/B0947BJ67M
8. ✓ https://www.amazon.com/Plugable-DisplayPort-Monitor-Universal-Ethernet/dp/B075DJMVW2
9. ✓ https://www.amazon.com/Plugable-Usb-Laptop-Docking-Station-Dual-Monitor-Hdmi-Ethernet/dp/B07CRSH25X


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


In [ ]:
# Cell 5: Test AmazonSearchAgent với keyword "laptop"
# 🔍 Trace sẽ được ghi lại tại: https://platform.openai.com/traces

# Khởi tạo agent
search_agent = AmazonSearchAgent()

# Tìm kiếm sản phẩm
keyword = "Laptop"
print(f"🔍 Searching for '{keyword}' on Amazon...")
print(f"   Trace: Check https://platform.openai.com/traces for details\n")

urls = search_agent.search(keyword, max_urls=20)

print(f"\n{'='*70}")
print(f"✅ Found {len(urls)} product URLs:")
print(f"{'='*70}\n")

for i, url in enumerate(urls, 1):
    # Highlight nếu URL có /dp/ (đúng format)
    is_valid = "/dp/" in url or "/gp/product/" in url
    status = "✓" if is_valid else "⚠️"
    print(f"{i}. {status} {url}")

INFO:root:[Amazon Search Agent] Initializing...
INFO:root:[Amazon Search Agent] Ready!
INFO:root:[Amazon Search Agent] Searching: Laptop


🔍 Searching for 'Laptop' on Amazon...
   Trace: Check https://platform.openai.com/traces for details



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:root:[Amazon Search Agent] Found 10 URLs



✅ Found 10 product URLs:

1. ✓ https://www.amazon.com/dp/B0F8HSTQJJ
2. ✓ https://www.amazon.com/dp/B0G3B5T7WB
3. ✓ https://www.amazon.com/dp/B09MDK7KF8
4. ✓ https://www.amazon.com/dp/B09478PVRL
5. ✓ https://www.amazon.com/dp/B0753GJBVW
6. ✓ https://www.amazon.com/dp/B0D3VMQ4MP
7. ✓ https://www.amazon.com/dp/B0917NMPCM
8. ✓ https://www.amazon.com/dp/B0026PPKSO
9. ✓ https://www.amazon.com/dp/B001MXJU5U
10. ✓ https://www.amazon.com/dp/B09HTTZJXJ


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


In [ ]:
search_agent = AmazonSearchAgent()
urls = search_agent.search("Smart TV", max_urls=20)
print(f"Found: {len(urls)} URLs")

INFO:root:[Amazon Search Agent] Initializing...
INFO:root:[Amazon Search Agent] Ready!
INFO:root:[Amazon Search Agent] Searching for: Smart TV
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
INFO:root:[Amazon Search Agent] Found 2 product URLs


Found: 2 URLs


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


In [8]:
# Cell 6: Amazon sale checker với việc set US Zip Code trước
from playwright.async_api import async_playwright
from typing import Tuple, List
import asyncio
import logging

logger = logging.getLogger(__name__)

# US Zip Code - California (same as your Windows setting)
US_ZIP_CODE = "96150"


async def set_amazon_us_location(page) -> bool:
    """
    Set Amazon delivery location to US by entering zip code.
    This needs to be done once before checking URLs.
    
    Args:
        page: Playwright page object
        
    Returns:
        True if successfully set location
    """
    try:
        print(f"📍 Setting Amazon location to US (Zip: {US_ZIP_CODE})...")
        
        # Go to Amazon homepage first
        await page.goto("https://www.amazon.com", timeout=30000, wait_until="domcontentloaded")
        await page.wait_for_timeout(2000)
        
        # Click on "Deliver to" location selector
        # This element is usually at top left, with id "nav-global-location-popover-link"
        location_btn = page.locator("#nav-global-location-popover-link")
        
        if await location_btn.count() > 0:
            await location_btn.click()
            await page.wait_for_timeout(1500)
            
            # Find zip code input field
            zip_input = page.locator('input[data-action="GLUXPostalInputAction"]')
            
            if await zip_input.count() > 0:
                # Clear and enter zip code
                await zip_input.fill(US_ZIP_CODE)
                await page.wait_for_timeout(500)
                
                # Click Apply button
                apply_btn = page.locator('input[aria-labelledby="GLUXZipUpdate-announce"]')
                if await apply_btn.count() > 0:
                    await apply_btn.click()
                    await page.wait_for_timeout(2000)
                    print(f"✅ Location set to US (Zip: {US_ZIP_CODE})")
                    return True
                else:
                    # Try alternative apply button
                    apply_btn2 = page.locator('span[data-action="GLUXPostalUpdateAction"] input')
                    if await apply_btn2.count() > 0:
                        await apply_btn2.click()
                        await page.wait_for_timeout(2000)
                        print(f"✅ Location set to US (Zip: {US_ZIP_CODE})")
                        return True
            else:
                # Maybe need to click "Change" first if already has location
                change_btn = page.locator('a[id="GLUXChangePostalCodeLink"]')
                if await change_btn.count() > 0:
                    await change_btn.click()
                    await page.wait_for_timeout(1000)
                    # Retry entering zip code
                    zip_input = page.locator('input[data-action="GLUXPostalInputAction"]')
                    if await zip_input.count() > 0:
                        await zip_input.fill(US_ZIP_CODE)
                        apply_btn = page.locator('input[aria-labelledby="GLUXZipUpdate-announce"]')
                        if await apply_btn.count() > 0:
                            await apply_btn.click()
                            await page.wait_for_timeout(2000)
                            print(f"✅ Location set to US (Zip: {US_ZIP_CODE})")
                            return True
        
        print("⚠️ Could not find location elements, but continuing...")
        return False
        
    except Exception as e:
        print(f"⚠️ Error setting location: {e}")
        return False


async def is_on_sale_amazon_playwright(url: str, page) -> Tuple[bool, dict]:
    """
    Check if Amazon product is on sale using Playwright.
    """
    try:
        await page.goto(url, timeout=30000, wait_until="domcontentloaded")
        await page.wait_for_timeout(2500)
        
        price_info = {}
        
        # Check sale indicator 1: savingsPercentage
        savings_elem = page.locator("span.savingsPercentage")
        has_savings = await savings_elem.count() > 0
        
        if has_savings:
            savings_text = await savings_elem.first.text_content()
            price_info["savings_pct"] = savings_text.strip()
        
        # Check sale indicator 2: basisPrice
        basis_elem = page.locator("span.basisPrice")
        has_basis = await basis_elem.count() > 0
        
        # Check sale indicator 3: data-a-strike
        strike_elem = page.locator('[data-a-strike="true"]')
        has_strike = await strike_elem.count() > 0
        
        # Get current price
        price_elem = page.locator("span.priceToPay")
        if await price_elem.count() > 0:
            price_text = await price_elem.first.text_content()
            price_info["sale_price"] = price_text.strip()
        
        is_sale = has_savings or has_basis or has_strike
        return is_sale, price_info
        
    except Exception as e:
        logger.warning(f"Error checking {url}: {e}")
        return False, {}


async def filter_amazon_sale_urls_playwright(urls: List[str]) -> List[Tuple[str, dict]]:
    """
    Filter Amazon URLs to keep only products on sale.
    Sets US location first, then checks each URL.
    """
    sale_items = []
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False,
            args=['--disable-blink-features=AutomationControlled', '--no-sandbox']
        )
        
        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
            viewport={'width': 1920, 'height': 1080},
            locale="en-US",
            timezone_id="America/New_York",
        )
        
        page = await context.new_page()
        
        # ===== STEP 1: Set US Location =====
        await set_amazon_us_location(page)
        
        # ===== STEP 2: Check each URL =====
        print(f"\n🔍 Checking {len(urls)} URLs for sale items...")
        print("="*60)
        
        for i, url in enumerate(urls, 1):
            is_sale, price_info = await is_on_sale_amazon_playwright(url, page)
            
            if is_sale:
                sale_items.append((url, price_info))
                savings = price_info.get("savings_pct", "N/A")
                price = price_info.get("sale_price", "N/A")
                print(f"[{i}/{len(urls)}] 🏷️ SALE | {savings} | {price}")
            else:
                price = price_info.get("sale_price", "No price found")
                print(f"[{i}/{len(urls)}] ⏭️ Skip | Price: {price}")
            
            await page.wait_for_timeout(500)
        
        await browser.close()
    
    print("="*60)
    print(f"✅ Found {len(sale_items)}/{len(urls)} products on sale")
    
    return sale_items


print("✅ Amazon sale checker with US ZIP code defined!")
print(f"📍 Will set location to ZIP: {US_ZIP_CODE} before checking")

✅ Amazon sale checker with US ZIP code defined!
📍 Will set location to ZIP: 96150 before checking


In [9]:
urls

['https://www.amazon.com/introducing-amazon-fire-tv-32-inch-2-series-hd-smart-tv/dp/B09N6F9NV3',
 'https://www.amazon.com/amazon-fire-tv-40-inch-2-series-hd-smart-tv/dp/B0CJCYMBZJ',
 'https://www.amazon.com/amazon-fire-tv-50-inch-4-series-4k-smart-tv/dp/B08SVZ775L',
 'https://www.amazon.com/amazon-fire-tv-32-inch-2-series-hd-smart-tv/dp/B0CJDSNN4T',
 'https://www.amazon.com/amazon-fire-tv-43-inch-4-series-4k-smart-tv/dp/B0CZ9WV2ZX',
 'https://www.amazon.com/INSIGNIA-24-inch-LED-FHD-Fire-TV/dp/B0FDYHLFMY',
 'https://www.amazon.com/amazon-fire-tv-50-inch-omni-series-4k-smart-tv/dp/B08T6F8YBH',
 'https://www.amazon.com/amazon-fire-tv-50-inch-4-series-4k-smart-tv/dp/B0B3GTSQ9Q']

In [10]:
# Cell 7: Test - xem browser set location thành công không

test_urls = [
    'https://www.amazon.com/Apple-2025-MacBook-13-inch-Laptop/dp/B0DZD9S5GC?th=1',  # SALE
    'https://www.amazon.com/amazon-fire-tv-55-inch-4-series-4k-smart-tv/dp/B0CZ9WV2ZX?th=1',  # NOT SALE
]

print("🚀 Testing with 2 known URLs...")
print("📍 Browser will set ZIP code to 96150 first\n")

sale_items = await filter_amazon_sale_urls_playwright(urls)

print(f"\n📋 Sale items found ({len(sale_items)}):")
for url, price_info in sale_items:
    print(f"   URL: {url[:60]}...")
    print(f"   Price info: {price_info}")

🚀 Testing with 2 known URLs...
📍 Browser will set ZIP code to 96150 first



📍 Setting Amazon location to US (Zip: 96150)...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


✅ Location set to US (Zip: 96150)

🔍 Checking 8 URLs for sale items...
[1/8] ⏭️ Skip | Price: $149.99
[2/8] ⏭️ Skip | Price: $249.99
[3/8] ⏭️ Skip | Price: No price found
[4/8] ⏭️ Skip | Price: $149.99
[5/8] 🏷️ SALE | -39% | $199.99
[6/8] 🏷️ SALE | -13% | $69.99
[7/8] ⏭️ Skip | Price: $407.99
[8/8] ⏭️ Skip | Price: No price found
✅ Found 2/8 products on sale

📋 Sale items found (2):
   URL: https://www.amazon.com/amazon-fire-tv-43-inch-4-series-4k-sm...
   Price info: {'savings_pct': '-39%', 'sale_price': '$199.99'}
   URL: https://www.amazon.com/INSIGNIA-24-inch-LED-FHD-Fire-TV/dp/B...
   Price info: {'savings_pct': '-13%', 'sale_price': '$69.99'}


In [11]:
# Cell 8: Define ScrapedAmazonDeal class
# Tương tự ScrapedBestBuyDeal nhưng cho Amazon

from typing import Optional
import re

class ScrapedAmazonDeal:
    """
    A class to represent a Deal scraped from Amazon using Playwright.
    This is the raw data before being processed by GPT.
    
    Attributes:
        title: Product title (max 200 chars)
        brand: Product brand (optional)
        price: Sale price in USD
        features: Product features text (max 1500 chars)
        url: Amazon product URL
    """
    
    title: str
    brand: Optional[str]
    price: float
    features: str
    url: str
    
    def __init__(
        self,
        title: str,
        brand: Optional[str],
        price: float,
        features: str,
        url: str
    ):
        """
        Initialize with scraped data from Amazon product page.
        
        Args:
            title: Product title
            brand: Product brand (optional)
            price: Sale price in USD
            features: Product features/description text
            url: Amazon product URL
        """
        self.title = title[:200] if title else "Unknown"
        self.brand = brand.strip() if brand else None
        self.price = price
        self.features = features[:1500] if features else ""
        self.url = url
    
    def __repr__(self) -> str:
        """Return a short string description."""
        return f"<{self.title[:50]}... | ${self.price}>"
    
    def describe(self) -> str:
        """
        Return a longer string to describe this deal for use in calling a model.
        Similar to ScrapedBestBuyDeal.describe() format.
        
        Returns:
            Formatted string with Title, Brand, Price, Features, URL
        """
        parts = [f"Title: {self.title}"]
        
        if self.brand:
            parts.append(f"Brand: {self.brand}")
        
        parts.append(f"Price: ${self.price:.2f}")
        
        if self.features and len(self.features) > 10:
            parts.append(f"Features: {self.features.strip()}")
        
        parts.append(f"URL: {self.url}")
        
        return "\n".join(parts)


print("✅ ScrapedAmazonDeal class defined!")
print("   Attributes: title, brand, price, features, url")
print("   Methods: __repr__(), describe()")

✅ ScrapedAmazonDeal class defined!
   Attributes: title, brand, price, features, url
   Methods: __repr__(), describe()


In [12]:
# Cell 9: scrape_amazon_products() với multi-selector fallback
# Scrape chi tiết sản phẩm Amazon từ các URLs đã filter (sale items)

import re
from typing import List, Tuple, Optional
from playwright.async_api import async_playwright

# US Zip Code (đã define ở Cell 6)
US_ZIP_CODE = "96150"


async def scrape_amazon_products(
    sale_items: List[Tuple[str, dict]],
    headless: bool = False
) -> List[ScrapedAmazonDeal]:
    """
    Scrape Amazon products and return as List[ScrapedAmazonDeal].
    
    Uses Playwright with multi-selector fallback strategy.
    
    Args:
        sale_items: List of (url, price_info) tuples from filter step
        headless: Run browser in headless mode (default: False)
        
    Returns:
        List[ScrapedAmazonDeal] - Raw scraped data from each product
    """
    scraped_deals = []
    
    # Multi-selector fallback lists
    TITLE_SELECTORS = [
        "#productTitle",
        "h1.product-title-word-break", 
        "h1 span#productTitle",
        "#title span"
    ]
    
    BRAND_SELECTORS = [
        "#bylineInfo",
        "a#bylineInfo", 
        "#brand",
        ".po-brand .a-span9 span",
        "a.contributorNameID"
    ]
    
    FEATURES_SELECTORS = [
        "#feature-bullets ul",
        "#featurebullets_feature_div ul",
        "#productDescription p",
        "#aplus-content-area",
        ".a-unordered-list.a-vertical"
    ]
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=headless,
            args=['--disable-blink-features=AutomationControlled', '--no-sandbox']
        )
        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36',
            viewport={'width': 1920, 'height': 1080},
            locale="en-US",
            timezone_id="America/New_York",
        )
        page = await context.new_page()
        
        # Set US location first
        await set_amazon_us_location(page)
        
        print(f"\n📦 Scraping {len(sale_items)} sale products...")
        print("="*60)
        
        for i, (url, price_info) in enumerate(sale_items, 1):
            print(f"[{i}/{len(sale_items)}] Scraping: {url[:60]}...")
            
            try:
                await page.goto(url, timeout=30000, wait_until="domcontentloaded")
                await page.wait_for_timeout(2500)
                
                # === EXTRACT TITLE (with fallback) ===
                title = "Unknown"
                for selector in TITLE_SELECTORS:
                    elem = page.locator(selector)
                    if await elem.count() > 0:
                        title = await elem.first.text_content()
                        title = title.strip() if title else "Unknown"
                        break
                
                # === EXTRACT BRAND (with fallback) ===
                brand = None
                for selector in BRAND_SELECTORS:
                    elem = page.locator(selector)
                    if await elem.count() > 0:
                        brand_text = await elem.first.text_content()
                        if brand_text:
                            # Clean up brand text (remove "Visit the X Store", "Brand: X")
                            brand = brand_text.strip()
                            brand = re.sub(r'^Visit the\s+', '', brand)
                            brand = re.sub(r'\s+Store$', '', brand)
                            brand = re.sub(r'^Brand:\s*', '', brand)
                        break
                
                # === EXTRACT FEATURES (with fallback) ===
                features = ""
                for selector in FEATURES_SELECTORS:
                    elem = page.locator(selector)
                    if await elem.count() > 0:
                        features = await elem.first.text_content()
                        features = features.strip() if features else ""
                        # Clean up features text
                        features = re.sub(r'\s+', ' ', features)
                        break
                
                # === EXTRACT PRICE (from price_info dict) ===
                price = 0.0
                price_text = price_info.get("sale_price", "$0")
                price_match = re.search(r'[\d,]+\.?\d*', price_text.replace(',', ''))
                if price_match:
                    price = float(price_match.group())
                
                # Create ScrapedAmazonDeal
                deal = ScrapedAmazonDeal(
                    title=title,
                    brand=brand,
                    price=price,
                    features=features,
                    url=url
                )
                scraped_deals.append(deal)
                
                # Log result
                print(f"    ✓ Title: {title[:50]}...")
                print(f"    ✓ Brand: {brand or 'N/A'}")
                print(f"    ✓ Price: ${price:.2f}")
                print(f"    ✓ Features: {len(features)} chars")
                
            except Exception as e:
                print(f"    ✗ Error: {e}")
                continue
        
        await browser.close()
    
    print("="*60)
    print(f"✅ Successfully scraped {len(scraped_deals)}/{len(sale_items)} products")
    
    return scraped_deals


print("✅ scrape_amazon_products() function defined!")
print("   Uses multi-selector fallback for: Title, Brand, Features")
print("   Price extracted from sale_items (Step 2)")

✅ scrape_amazon_products() function defined!
   Uses multi-selector fallback for: Title, Brand, Features
   Price extracted from sale_items (Step 2)


In [13]:
# Cell 10: Test scrape_amazon_products() với sale_items từ Step 2
# sale_items là kết quả từ filter_amazon_sale_urls_playwright()

print(f"📋 Sale items to scrape: {len(sale_items)}")
for i, (url, info) in enumerate(sale_items, 1):
    print(f"   {i}. {url[:60]}... | {info}")

print("\n🚀 Starting scrape...")

# Scrape products
scraped_deals = await scrape_amazon_products(sale_items, headless=False)

# Display results
print(f"\n{'='*70}")
print(f"📦 SCRAPED DEALS ({len(scraped_deals)}):")
print(f"{'='*70}\n")

for i, deal in enumerate(scraped_deals, 1):
    print(f"--- Deal {i} ---")
    print(deal.describe())
    print()

📋 Sale items to scrape: 2
   1. https://www.amazon.com/amazon-fire-tv-43-inch-4-series-4k-sm... | {'savings_pct': '-39%', 'sale_price': '$199.99'}
   2. https://www.amazon.com/INSIGNIA-24-inch-LED-FHD-Fire-TV/dp/B... | {'savings_pct': '-13%', 'sale_price': '$69.99'}

🚀 Starting scrape...
📍 Setting Amazon location to US (Zip: 96150)...
✅ Location set to US (Zip: 96150)

📦 Scraping 2 sale products...
[1/2] Scraping: https://www.amazon.com/amazon-fire-tv-43-inch-4-series-4k-sm...
    ✓ Title: Amazon Fire TV 43" 4-Series 4K UHD smart TV, strea...
    ✓ Brand: Amazon Fire TV
    ✓ Price: $199.99
    ✓ Features: 1018 chars
[2/2] Scraping: https://www.amazon.com/INSIGNIA-24-inch-LED-FHD-Fire-TV/dp/B...
    ✓ Title: INSIGNIA 24" Class F40 Series LED Full HD Smart Fi...
    ✓ Brand: INSIGNIA
    ✓ Price: $69.99
    ✓ Features: 1580 chars
✅ Successfully scraped 2/2 products

📦 SCRAPED DEALS (2):

--- Deal 1 ---
Title: Amazon Fire TV 43" 4-Series 4K UHD smart TV, stream live TV without cable, 202

In [14]:
# Cell 11: AmazonScannerAgent - Chọn top 5 deals từ scraped data
# Tương tự BestBuyScannerAgent, sử dụng GPT-5-mini với Structured Outputs

from openai import OpenAI
from typing import Optional, List
from pydantic import BaseModel, Field

# Import Deal và DealSelection từ price_agents (đã có sẵn)
from price_agents.deals import Deal, DealSelection


class AmazonScannerAgent:
    """
    Agent that uses GPT-5-mini to select the best 5 deals from scraped Amazon products.
    
    This agent:
    1. Takes a list of ScrapedAmazonDeal objects
    2. Uses GPT-5-mini with structured outputs
    3. Returns a DealSelection with the best 5 deals
    """
    
    name = "Amazon Scanner Agent"
    MODEL = "gpt-5-mini"
    
    SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
    Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description.
    Most important is that you respond with the 5 deals that have the most detailed product description with price.
    
    **IMPORTANT:**
    1. Focus on the product features and specifications, not sales terms.
    2. The product_description should be a 3-4 sentence summary of the product itself.
    3. Price must be greater than 0.
    4. Keep the original URL exactly as provided.
    """
    
    USER_PROMPT_PREFIX = """Respond with the most promising deals from this list (up to 5), selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
    You should rephrase the description to be a summary of the product itself, not the terms of the deal.
    Remember to respond with a short paragraph of text in the product_description field for each item that you select.
    
    Deals:
    
    """
    
    USER_PROMPT_SUFFIX = "\n\nInclude up to 5 deals, no more."
    
    def __init__(self):
        """Initialize with OpenAI client."""
        print(f"[{self.name}] Initializing...")
        self.openai = OpenAI()
        print(f"[{self.name}] Ready!")
    
    def make_user_prompt(self, scraped_deals: List[ScrapedAmazonDeal]) -> str:
        """
        Create user prompt from scraped deals.
        
        Args:
            scraped_deals: List of ScrapedAmazonDeal objects
            
        Returns:
            Formatted prompt string for GPT
        """
        user_prompt = self.USER_PROMPT_PREFIX
        user_prompt += "\n\n".join([deal.describe() for deal in scraped_deals])
        user_prompt += self.USER_PROMPT_SUFFIX
        return user_prompt
    
    def scan(self, scraped_deals: List[ScrapedAmazonDeal]) -> Optional[DealSelection]:
        """
        Call GPT-5-mini to select the best deals with good descriptions and prices.
        
        Uses OpenAI structured outputs to ensure response conforms to DealSelection schema.
        
        Args:
            scraped_deals: List of ScrapedAmazonDeal from Playwright scraping
            
        Returns:
            DealSelection with up to 5 best deals, or None if no valid deals
        """
        if not scraped_deals:
            print(f"[{self.name}] No deals to scan")
            return None
        
        # Filter deals with price > 0
        valid_deals = [d for d in scraped_deals if d.price > 0]
        if not valid_deals:
            print(f"[{self.name}] No deals with valid price > 0")
            return None
        
        user_prompt = self.make_user_prompt(valid_deals)
        
        print(f"[{self.name}] Calling {self.MODEL} with {len(valid_deals)} deals...")
        
        result = self.openai.chat.completions.parse(
            model=self.MODEL,
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            response_format=DealSelection,
        )
        
        selection = result.choices[0].message.parsed
        
        # Filter out any deals with price <= 0
        selection.deals = [deal for deal in selection.deals if deal.price > 0]
        
        print(f"[{self.name}] Selected {len(selection.deals)} deals")
        
        return selection


print("✅ AmazonScannerAgent class defined!")
print("   Model: GPT-5-mini")
print("   Output: DealSelection (reuse from price_agents.deals)")

✅ AmazonScannerAgent class defined!
   Model: GPT-5-mini
   Output: DealSelection (reuse from price_agents.deals)


In [15]:
# Cell 12: Test AmazonScannerAgent với scraped_deals từ Step 3

print(f"📋 Scraped deals to scan: {len(scraped_deals)}")
for i, deal in enumerate(scraped_deals, 1):
    print(f"   {i}. {deal.title[:50]}... | ${deal.price}")

print("\n🤖 Running AmazonScannerAgent...")

# Initialize agent
scanner = AmazonScannerAgent()

# Scan deals
deal_selection = scanner.scan(scraped_deals)

# Display results
if deal_selection and deal_selection.deals:
    print(f"\n{'='*70}")
    print(f"🏆 TOP DEALS SELECTED ({len(deal_selection.deals)}):")
    print(f"{'='*70}\n")
    
    for i, deal in enumerate(deal_selection.deals, 1):
        print(f"--- Deal {i} ---")
        print(f"Description: {deal.product_description[:200]}...")
        print(f"Price: ${deal.price:.2f}")
        print(f"URL: {deal.url}")
        print()
else:
    print("❌ No deals selected")

📋 Scraped deals to scan: 2
   1. Amazon Fire TV 43" 4-Series 4K UHD smart TV, strea... | $199.99
   2. INSIGNIA 24" Class F40 Series LED Full HD Smart Fi... | $69.99

🤖 Running AmazonScannerAgent...
[Amazon Scanner Agent] Initializing...
[Amazon Scanner Agent] Ready!
[Amazon Scanner Agent] Calling gpt-5-mini with 2 deals...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[Amazon Scanner Agent] Selected 2 deals

🏆 TOP DEALS SELECTED (2):

--- Deal 1 ---
Description: A 43-inch Amazon-built smart TV featuring 4K Ultra HD resolution with HDR10 and HLG support for improved contrast and color, paired with Dolby Digital Plus audio. The set has an ultra-slim bezel and r...
Price: $199.99
URL: https://www.amazon.com/amazon-fire-tv-43-inch-4-series-4k-smart-tv/dp/B0CZ9WV2ZX

--- Deal 2 ---
Description: A 24-inch Full HD (1080p) Insignia LED smart TV that uses the Fire TV experience to provide access to apps such as Prime Video, Netflix, Disney+, Hulu, and a range of ad-supported streaming channels. ...
Price: $69.99
URL: https://www.amazon.com/INSIGNIA-24-inch-LED-FHD-Fire-TV/dp/B0FDYHLFMY



In [16]:
# Cell 13: Initialize EnsembleAgent
# Load ChromaDB và tạo EnsembleAgent (3 models: Frontier + Specialist + Neural)

import chromadb
from price_agents.ensemble_agent import EnsembleAgent
from price_agents.deals import Opportunity

# Path to ChromaDB
DB_PATH = "products_vectorstore"

print("📊 Initializing ChromaDB...")
client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection('products')
print(f"   Collection: {collection.name}")
print(f"   Documents: {collection.count()}")

print("\n🤖 Initializing EnsembleAgent (3 models)...")
ensemble = EnsembleAgent(collection)
print("✅ EnsembleAgent ready!")

INFO:datasets:PyTorch version 2.9.0 available.
INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


📊 Initializing ChromaDB...


INFO:root:[Ensemble Agent] Initializing Ensemble Agent
INFO:root:[Specialist Agent] Specialist Agent is initializing - connecting to modal
INFO:root:[Specialist Agent] Specialist Agent is ready
INFO:root:[Frontier Agent] Initializing Frontier Agent
INFO:root:[Frontier Agent] Frontier Agent is setting up with OpenAI


   Collection: products
   Documents: 800000

🤖 Initializing EnsembleAgent (3 models)...


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:root:[Frontier Agent] Frontier Agent is ready
INFO:root:[Neural Network Agent] Neural Network Agent is initializing
INFO:root:Neural Network is using cuda
INFO:root:[Neural Network Agent] Neural Network Agent is ready and weights are loaded
INFO:root:[Ensemble Agent] Ensemble Agent is ready


✅ EnsembleAgent ready!


In [17]:
# Cell 14: Estimate prices với EnsembleAgent và tính discount

print(f"💰 Estimating prices for {len(deal_selection.deals)} deals...")
print("   Using: Frontier (80%) + Specialist (10%) + Neural (10%)")
print("="*60)

opportunities = []

for i, deal in enumerate(deal_selection.deals, 1):
    print(f"\n[{i}/{len(deal_selection.deals)}] {deal.product_description[:50]}...")
    
    # Estimate true value với EnsembleAgent
    estimate = ensemble.price(deal.product_description)
    
    # Calculate discount
    discount = estimate - deal.price
    discount_pct = (discount / estimate * 100) if estimate > 0 else 0
    
    # Create Opportunity
    opp = Opportunity(
        deal=deal,
        estimate=estimate,
        discount=discount
    )
    opportunities.append(opp)
    
    # Status icon
    if discount > 100:
        status = "🔥 HOT DEAL!"
    elif discount > 50:
        status = "✅ Good deal"
    elif discount > 0:
        status = "👍 OK"
    else:
        status = "❌ Overpriced"
    
    print(f"    Sale Price:  ${deal.price:.2f}")
    print(f"    Est. Value:  ${estimate:.2f}")
    print(f"    Discount:    ${discount:.2f} ({discount_pct:.1f}%) {status}")

# Sort by discount (highest first)
opportunities.sort(key=lambda x: x.discount, reverse=True)

print(f"\n{'='*60}")
print(f"🏆 FINAL RESULTS (sorted by discount):")
print(f"{'='*60}\n")

for i, opp in enumerate(opportunities, 1):
    discount_pct = (opp.discount / opp.estimate * 100) if opp.estimate > 0 else 0
    print(f"{i}. {opp.deal.product_description[:40]}...")
    print(f"   💵 ${opp.deal.price:.2f} → Est: ${opp.estimate:.2f} = Save ${opp.discount:.2f} ({discount_pct:.1f}%)")
    print(f"   🔗 {opp.deal.url}")
    print()

INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
09:42:54 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


💰 Estimating prices for 2 deals...
   Using: Frontier (80%) + Specialist (10%) + Neural (10%)

[1/2] A 43-inch Amazon-built smart TV featuring 4K Ultra...


09:42:55 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $220.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $289.99
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $325.53
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $286.54
INFO:root:[Ensemble Agent] Running Ensemble Agent - preprocessing text
09:43:53 - LiteLLM:INFO: utils.py:3421 - 
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq
INFO:LiteLLM:
LiteLLM completion() model= openai/gpt-oss-20b; provider = groq


    Sale Price:  $199.99
    Est. Value:  $286.54
    Discount:    $86.55 (30.2%) ✅ Good deal

[2/2] A 24-inch Full HD (1080p) Insignia LED smart TV th...


09:43:53 - LiteLLM:INFO: utils.py:1302 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Ensemble Agent] Pre-processed text using groq/openai/gpt-oss-20b
INFO:root:[Specialist Agent] Specialist Agent is calling remote fine-tuned model
INFO:root:[Specialist Agent] Specialist Agent completed - predicting $130.00
INFO:root:[Frontier Agent] Frontier Agent is performing a RAG search of the Chroma datastore to find 5 similar products


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:[Frontier Agent] Frontier Agent has found similar products
INFO:root:[Frontier Agent] Frontier Agent is about to call gpt-5.1 with context including 5 similar products
INFO:root:[Frontier Agent] Frontier Agent completed - predicting $129.99
INFO:root:[Neural Network Agent] Neural Network Agent is starting a prediction
INFO:root:[Neural Network Agent] Neural Network Agent completed - predicting $94.37
INFO:root:[Ensemble Agent] Ensemble Agent complete - returning $126.43


    Sale Price:  $69.99
    Est. Value:  $126.43
    Discount:    $56.44 (44.6%) ✅ Good deal

🏆 FINAL RESULTS (sorted by discount):

1. A 43-inch Amazon-built smart TV featurin...
   💵 $199.99 → Est: $286.54 = Save $86.55 (30.2%)
   🔗 https://www.amazon.com/amazon-fire-tv-43-inch-4-series-4k-smart-tv/dp/B0CZ9WV2ZX

2. A 24-inch Full HD (1080p) Insignia LED s...
   💵 $69.99 → Est: $126.43 = Save $56.44 (44.6%)
   🔗 https://www.amazon.com/INSIGNIA-24-inch-LED-FHD-Fire-TV/dp/B0FDYHLFMY

